# 02 — Data Cleaning
**CareFlow Process Mining | Google Colab**

Creates a separate cleaned event log from the processed workbook. The source upload is not modified.
Default policy: trim text whitespace, parse timestamps, remove exact duplicate rows, and remove rows missing a case ID or activity because those rows cannot reliably participate in case-sequence analysis. Review the cleaning summary before relying on the output.

In [ ]:
!pip -q install openpyxl pandas numpy matplotlib seaborn pm4py

import io, os, re, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
print("CareFlow Process Mining environment ready.")

In [ ]:
# Upload: Sepsis_Cases_Event_Log_Cleaned_for_Visualization.xlsx
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload the processed sepsis workbook.")
workbook_name = next(iter(uploaded))
workbook_bytes = uploaded[workbook_name]
xls = pd.ExcelFile(io.BytesIO(workbook_bytes))
print("Workbook:", workbook_name)
print("Sheets:", xls.sheet_names)

required_sheet = "Event Log Visual"
if required_sheet not in xls.sheet_names:
    raise ValueError(f"Expected sheet '{required_sheet}'. Available sheets: {xls.sheet_names}")

events = pd.read_excel(io.BytesIO(workbook_bytes), sheet_name="Event Log Visual")
events.columns = [str(c).strip() for c in events.columns]
print("Event Log Visual shape:", events.shape)
display(events.head())

In [ ]:
# This workbook's event-log sheet uses these verified column names:
CASE_COL = "case_id"
ACTIVITY_COL = "activity"
TIME_COL = "timestamp"
EVENT_INDEX_COL = "event_index"
ORG_COL = "org_group"
LIFECYCLE_COL = "lifecycle_transition"

# Parse timestamp safely; invalid values become NaT and are reported.
if TIME_COL in events.columns:
    events[TIME_COL] = pd.to_datetime(events[TIME_COL], errors="coerce")

print("Case column:", CASE_COL, "| present:", CASE_COL in events.columns)
print("Activity column:", ACTIVITY_COL, "| present:", ACTIVITY_COL in events.columns)
print("Timestamp column:", TIME_COL, "| present:", TIME_COL in events.columns)

## 1. Apply documented cleaning rules

In [ ]:
clean = events.copy()
rows_before = len(clean)

# Trim whitespace from text fields without converting missing values to strings.
for col in clean.select_dtypes(include=["object", "string"]).columns:
    clean[col] = clean[col].map(lambda value: value.strip() if isinstance(value, str) else value)

# Parse timestamps; invalid values are retained as missing so they can be reviewed.
clean[TIME_COL] = pd.to_datetime(clean[TIME_COL], errors="coerce")

duplicates_found = int(clean.duplicated().sum())
clean = clean.drop_duplicates().copy()
rows_after_dedup = len(clean)

missing_case_before = int(clean[CASE_COL].isna().sum())
missing_activity_before = int(clean[ACTIVITY_COL].isna().sum())
clean = clean.dropna(subset=[CASE_COL, ACTIVITY_COL]).copy()
rows_after_required_fields = len(clean)

# Sort for reproducible downstream process analysis.
sort_cols = [CASE_COL]
if EVENT_INDEX_COL in clean.columns:
    sort_cols.append(EVENT_INDEX_COL)
if TIME_COL in clean.columns:
    sort_cols.append(TIME_COL)
clean = clean.sort_values(sort_cols, na_position="last").reset_index(drop=True)

cleaning_summary = pd.DataFrame([
    ["Rows in input event log", rows_before],
    ["Exact duplicate rows found", duplicates_found],
    ["Rows after exact duplicate removal", rows_after_dedup],
    ["Rows missing case_id before required-field removal", missing_case_before],
    ["Rows missing activity before required-field removal", missing_activity_before],
    ["Rows after removing missing case_id/activity", rows_after_required_fields],
    ["Rows removed overall", rows_before - rows_after_required_fields],
    ["Rows with missing/unparseable timestamp retained for review", int(clean[TIME_COL].isna().sum())],
], columns=["metric", "value"])
display(cleaning_summary)
display(clean.head())

## 2. Review remaining missingness

In [ ]:
missing_after = pd.DataFrame({
    "missing_count": clean.isna().sum(),
    "missing_percent": (clean.isna().mean()*100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing_after)
print("Do not replace missing clinical measurements with zero. Missing and zero are not equivalent.")

## 3. Save cleaned event log and audit files

In [ ]:
clean.to_csv("sepsis_event_log_cleaned.csv", index=False)
cleaning_summary.to_csv("cleaning_summary.csv", index=False)
missing_after.to_csv("missing_values_after_cleaning.csv")
files.download("sepsis_event_log_cleaned.csv")
files.download("cleaning_summary.csv")
files.download("missing_values_after_cleaning.csv")

## Cleaning decisions to document
- Exact duplicate removal is applied to identical full rows only.
- Rows missing `case_id` or `activity` are excluded from the process-analysis output.
- Rows with missing/unparseable timestamps are retained, but timestamp-based analyses must exclude or separately investigate them.
- Always keep the uploaded original workbook unchanged.